# mmBERT-base 전체 대화 파인튜닝 — 긴 입력은 구간별 인코딩 후 통합

**새 A100/H100 GPU Colab 런타임**에서 실행하세요. 고용량 RAM 권장.
이 노트북은 연구용 구현이며 실제 A100/H100 학습을 개발 환경에서 실행하지 못했습니다.
Qwen 라벨 생성·CoVe 검증·CaMVo 투표를 다시 실행하지 않습니다. vLLM이나 임베딩 준비도 필요 없습니다.

## 데이터와 목표

Drive/ThinkGuard_NLP/studychat_qwen_blind_verify_v1/final/{train,valid,test}_ok.csv를 사용합니다.
앞서 정리한 splits_v2_qwen_verified와 동일한 원본 파일입니다. 기본 8,706/1,176/1,081건.
이전에 업로드한 thinkguard_qwen_verification_bundle.zip에서 원문만 읽습니다. 새 ZIP은 필요 없습니다.
ID와 원문 SHA로 연결하고, 과거 모델 라벨·검증 설명은 모델 입력에서 제외합니다.
학습 목표: requests_complete_answer / has_self_attempt_evidence 두 개. learning_context는 입력으로 주지 않습니다.
한 샘플은 현재 학생 발화 한 건과 그 시점 이전 대화입니다. 전체 chat당 라벨 하나를 붙이는 것이 아닙니다.
원본 파일을 덮어쓰지 않고 새 실험 폴더에 입력 스냅샷·결과를 저장합니다.

## 8,192토큰 초과 처리

- 원문 전체를 역할/발화 번호로 표시하고, 최대 8,192토큰 구간으로 나눕니다. 구간은 256토큰 겹칩니다.
- 같은 mmBERT를 모든 구간에 적용합니다. MAX_CHUNKS·앞/뒤 자르기·검색에 따른 생략이 없습니다.
- 현재 발화 자체가 길어도 모두 처리합니다. query는 현재 학생 토큰 전체에서 만듭니다.
- 구간 순서·현재 발화 위치·학생 토큰 비율을 활용해 현재 query에 조건화한 attention으로 합칩니다.
- 자기 시도 경로는 학생 토큰을 pooling합니다. AI 토큰은 문맥으로는 읽습니다.
- 구간마다 라벨을 복제하지 않습니다. **현재 학생 발화 샘플당 하나의 두 라벨 BCE 손실**을 계산합니다.
- overlap 토큰은 문맥으로 중복 읽지만 pooling 소유권은 한 번만 줍니다.

이는 8,192 위치 한도를 늘린 모델이 아닙니다. 전체 토큰을 보는 계층형 근사이며,
모든 토큰 사이의 전역 self-attention과 같지 않습니다. 경계를 가로지르는 추론이 약해질 수 있습니다.
학생 토큰 표현에도 주변 AI 문맥이 섞입니다. 역할 pooling·attention은 의미적 근거 인증이 아닙니다.
같은 과제인지, 학생이 실제 시도했는지는 여전히 학습·평가 대상입니다.

## 학습·메모리·재개

307M 인코더와 새 분류기를 **전체 파인튜닝**합니다. LoRA가 아닙니다.
BF16 autocast, 구간 microbatch, activation checkpoint로 메모리를 줄입니다.
학습의 batch는 추론 동시 요청과 다릅니다. OOM이면 먼저 WINDOW_BATCH/RECORD_BATCH를 낮추세요.
이 두 값과 SAVE_STEPS는 같은 RUN_NAME에서 변경 가능합니다. 숫자 연산/RNG 경로는 달라질 수 있습니다.
CHUNK_LENGTH·학습률·라벨·모델·epoch 변경은 새 RUN_NAME을 사용합니다.
전체 optimizer 체크포인트는 수 GB입니다. latest와 최대 3개 best 가중치 및 토큰 캐시용 Drive 공간이 필요합니다.
SAVE_STEPS=100과 epoch 종료에 원자적으로 저장하며 마지막 미저장 step은 중단 후 다시 계산합니다.
동일 환경·설정에서 optimizer/RNG/다음 대화 위치를 복원합니다. GPU가 바뀌면 비트 단위 동일성을 보장하지 않습니다.

## 평가 규칙

train만 학습·클래스 가중치 계산에 사용합니다. valid만 checkpoint와 두 threshold 선택에 사용합니다.
선택 지표는 각 이진 라벨의 class0/1 Macro-F1을 구한 뒤 두 라벨을 평균한 값입니다.
test는 마지막 별도 셀에서만 예측합니다. 높은 점수는 자동 검증 Qwen 라벨과의 일치이며 실제 정답률 증명이 아닙니다.
분할 간 정확히 같은 원문 중복은 test>valid>train 우선순위로 제외하고 ID/사유를 기록합니다.
분할 내 같은 원문도 첫 ID만 유지하므로 실제 학습 건수는 위 CSV 건수보다 줄 수 있습니다.

근거: [mmBERT 논문](https://proceedings.mlr.press/v306/marone26a.html),
[공식 모델](https://huggingface.co/jhu-clsp/mmBERT-base),
[계층형 Attention 원리, NAACL 2016](https://aclanthology.org/N16-1174/).
본 구조는 이 원리를 대화에 응용한 것으로 해당 논문의 완전 재현은 아닙니다.


## 1. Drive·기존 CSV/원문 스냅샷

In [ ]:
import hashlib
import json
import shutil
import zipfile
from pathlib import Path
from google.colab import drive, files

drive.mount('/content/drive')
PROJECT = Path('/content/drive/MyDrive/ThinkGuard_NLP')
RUN_NAME = 'studychat_mmbert_hierarchical_v1'
RUN_DIR = PROJECT / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)
# 정리한 공유 폴더와 내용이 같은 원래 export 폴더를 기본 경로로 사용합니다.
SPLITS_DIR = PROJECT / 'studychat_qwen_blind_verify_v1' / 'final'
DATA_DIR = RUN_DIR / 'inputs'
DATA_DIR.mkdir(exist_ok=True)
for split in ('train','valid','test'):
    name = f'{split}_ok.csv'
    source, target = SPLITS_DIR/name, DATA_DIR/name
    if not source.exists():
        raise FileNotFoundError('CSV가 없습니다. SPLITS_DIR을 확인하세요: '+str(source))
    if target.exists():
        if hashlib.sha256(source.read_bytes()).digest() != hashlib.sha256(target.read_bytes()).digest():
            raise ValueError('원본 CSV가 바뀌었습니다. RUN_NAME을 새로 지정하세요.')
    else:
        shutil.copy2(source,target)
BUNDLE_NAME = 'thinkguard_qwen_verification_bundle.zip'
BUNDLE = PROJECT/BUNDLE_NAME
if not BUNDLE.exists():
    print('이전에 제공한 검증용 ZIP을 업로드하세요:',BUNDLE_NAME)
    uploaded = files.upload()
    if set(uploaded) != {BUNDLE_NAME}:
        raise ValueError('검증용 ZIP 하나를 선택하세요.')
    BUNDLE.write_bytes(uploaded[BUNDLE_NAME])
    del uploaded
with zipfile.ZipFile(BUNDLE) as archive:
    manifest = json.loads(archive.read('bundle_manifest.json'))
    expected = manifest['files']['inputs.jsonl.gz']
    target = DATA_DIR/'inputs.jsonl.gz'
    if not target.exists():
        with archive.open('inputs.jsonl.gz') as src, target.with_suffix('.tmp').open('wb') as dst:
            shutil.copyfileobj(src,dst)
        target.with_suffix('.tmp').replace(target)
    with target.open('rb') as stream:
        if hashlib.file_digest(stream,'sha256').hexdigest() != expected:
            raise ValueError('원문 해시 불일치')
print('입력 스냅샷:',DATA_DIR)
print('모델·결과 저장:',RUN_DIR)


## 2. 학습 전용 환경 설치·버전 고정

In [ ]:
import importlib.metadata
import subprocess
import sys

PACKAGES = ['torch','transformers','huggingface-hub','numpy','safetensors','sentencepiece']
lock = RUN_DIR/'environment_versions.json'
if lock.exists():
    pinned = json.loads(lock.read_text(encoding='utf-8'))
    requirements = [f'{name}=={pinned[name]}' for name in PACKAGES]
else:
    # Colab의 CUDA PyTorch는 유지. vLLM 설치와 섞지 말고 새 GPU 런타임에서 실행하세요.
    requirements = ['transformers>=4.56,<5','huggingface-hub>=0.34,<1',
                    'numpy>=2,<3','safetensors','sentencepiece']
subprocess.check_call([sys.executable,'-m','pip','install','-q','--upgrade',*requirements])
VERSIONS = {name:importlib.metadata.version(name) for name in PACKAGES}
lock.write_text(json.dumps(VERSIONS,indent=2),encoding='utf-8')
print('환경 고정:',VERSIONS)
print('GPU 모델은 자식 프로세스에서 로드하므로 이 셀에서는 GPU 메모리를 사용하지 않습니다.')


## 3. 내장 코드 준비

In [ ]:
import sys
from pathlib import Path
CODE = Path('/content/thinkguard_mmbert_training_code')
CODE.mkdir(exist_ok=True)
SOURCES = {'mmtrain/__init__.py': '"""Hierarchical mmBERT training on full source conversations and two weak labels."""\n', 'mmtrain/checkpoint.py': '"""Atomic Drive checkpoints with optimizer and RNG state, only at optimizer boundaries."""\n\nimport os\nfrom pathlib import Path\n\nimport torch\n\n\ndef save(path, state):\n    path = Path(path)\n    temporary = path.with_suffix(\'.tmp\')\n    with temporary.open(\'wb\') as stream:\n        torch.save(state,stream)\n        stream.flush()\n        os.fsync(stream.fileno())\n    temporary.replace(path)\n\n\ndef rng_state():\n    return {\'cpu\':torch.get_rng_state(), \'cuda\':torch.cuda.get_rng_state_all() if torch.cuda.is_available() else []}\n\n\ndef restore_rng(state):\n    torch.set_rng_state(state[\'cpu\'])\n    if state[\'cuda\'] and torch.cuda.is_available():\n        torch.cuda.set_rng_state_all(state[\'cuda\'])\n\n\ndef load(path, identity):\n    value = torch.load(path,map_location=\'cpu\',weights_only=True)\n    if value[\'identity\'] != identity:\n        raise ValueError(\'Checkpoint config/model/data/code identity differs\')\n    return value\n', 'mmtrain/chunks.py': '"""Tokenize all source turns; overlapping context, disjoint ownership for pooling."""\n\n\ndef special_wrapper(tokenizer):\n    probe = \'A short tokenizer probe.\'\n    plain = tokenizer.encode(probe, add_special_tokens=False)\n    wrapped = tokenizer(probe, add_special_tokens=True, return_special_tokens_mask=True)\n    ids, masks = wrapped[\'input_ids\'], wrapped[\'special_tokens_mask\']\n    indexes = [i for i,m in enumerate(masks) if not m]\n    if not indexes or indexes != list(range(indexes[0],indexes[-1]+1)):\n        raise ValueError(\'Unsupported noncontiguous special-token template\')\n    if ids[indexes[0]:indexes[-1]+1] != plain:\n        raise ValueError(\'Special-token template changed input content\')\n    return ids[:indexes[0]], ids[indexes[-1]+1:]\n\n\ndef make_chunks(source, tokenizer, length=4096, overlap=256):\n    if not 16 <= length <= 8192:\n        raise ValueError(\'Chunk length must be between 16 and 8192\')\n    prefix, suffix = special_wrapper(tokenizer)\n    capacity = length-len(prefix)-len(suffix)\n    if not 0 <= overlap < capacity:\n        raise ValueError(\'Overlap must be smaller than content capacity\')\n    tokens, student, current = [], [], []\n    turns = [*enumerate(source[\'history\']), (-1, {\'role\':\'student\',\'content\':source[\'message\']})]\n    for index, turn in turns:\n        if turn[\'role\'] not in (\'student\',\'assistant\') or not isinstance(turn[\'content\'],str):\n            raise ValueError(\'Invalid source role/content\')\n        marker = f\'\\n[TURN={index}; ROLE={turn["role"]}; CURRENT={index == -1}]\\n\'\n        meta = tokenizer.encode(marker, add_special_tokens=False, truncation=False, verbose=False)\n        text = tokenizer.encode(turn[\'content\'], add_special_tokens=False, truncation=False, verbose=False)\n        tokens.extend(meta+text)\n        student.extend([0]*len(meta)+[int(turn[\'role\']==\'student\')]*len(text))\n        current.extend([0]*len(meta)+[int(index==-1)]*len(text))\n    if not tokens:\n        raise ValueError(\'Empty serialized conversation\')\n    chunks, start, owned = [], 0, 0\n    while start < len(tokens):\n        end = min(start+capacity,len(tokens))\n        skip = 0 if not chunks else overlap\n        own = [0]*skip+[1]*(end-start-skip)\n        def mask(values,start=start,end=end,own=own):\n            return [0]*len(prefix)+[v*o for v,o in zip(values[start:end],own,strict=True)]+[0]*len(suffix)\n        chunks.append({\'input_ids\':prefix+tokens[start:end]+suffix,\n                       \'student_mask\':mask(student), \'current_mask\':mask(current),\n                       \'owner_mask\':[0]*len(prefix)+own+[0]*len(suffix),\n                       \'start\':start,\'end\':end})\n        owned += sum(own)\n        if end == len(tokens):\n            break\n        start = end-overlap\n    if owned != len(tokens) or sum(sum(c[\'current_mask\']) for c in chunks) != sum(current):\n        raise ValueError(\'Token coverage or overlap ownership failure\')\n    return chunks, {\'tokens\':len(tokens),\'single_input_tokens\':len(tokens)+len(prefix)+len(suffix),\n                    \'chunks\':len(chunks), \'current_tokens\':sum(current),\n                    \'student_tokens\':sum(student),\'encoded_tokens\':sum(len(c[\'input_ids\']) for c in chunks)}\n', 'mmtrain/data.py': '"""Frozen CSV IDs, provenance and group leakage checks; labels never enter text."""\n\nimport csv\nimport gzip\nimport hashlib\nimport json\nfrom pathlib import Path\n\nLABELS = (\'requests_complete_answer\',\'has_self_attempt_evidence\')\n\n\ndef sha(path):\n    with Path(path).open(\'rb\') as stream:\n        return hashlib.file_digest(stream,\'sha256\').hexdigest()\n\n\ndef stable(value):\n    return hashlib.sha256(json.dumps(value,sort_keys=True,ensure_ascii=False).encode()).hexdigest()\n\n\ndef load_sources(directory):\n    directory = Path(directory)\n    labels, splits, students, chats = {}, {}, {}, {}\n    for split in (\'train\',\'valid\',\'test\'):\n        with (directory/f\'{split}_ok.csv\').open(encoding=\'utf-8-sig\',newline=\'\') as stream:\n            rows = list(csv.DictReader(stream))\n        splits[split] = []\n        for row in rows:\n            uid = row[\'dataset\']+\':\'+row[\'source_id\']\n            if row[\'dataset\'] != \'studychat\' or uid in labels or row[\'status\'] != \'ok\':\n                raise ValueError(\'Duplicate ID or non-ok StudyChat row\')\n            if any(row[k] not in (\'true\',\'false\') for k in (*LABELS,\'is_learning_context\')):\n                raise ValueError(\'Null or malformed label in _ok CSV\')\n            for key, owners in ((\'student_id\',students),(\'conversation_id\',chats)):\n                if not row[key] or owners.setdefault(row[key],split) != split:\n                    raise ValueError(\'Student/conversation overlaps splits\')\n            labels[uid] = row\n            splits[split].append(uid)\n    sources = {}\n    with gzip.open(directory/\'inputs.jsonl.gz\',\'rt\',encoding=\'utf-8\') as stream:\n        for line in stream:\n            row = json.loads(line)\n            uid = row[\'dataset\']+\':\'+row[\'source_id\']\n            if uid not in labels:\n                continue\n            if uid in sources or row.get(\'input_verified\') is not True:\n                raise ValueError(\'Duplicate or unverified original\')\n            if stable({k:row[k] for k in (\'message\',\'history\')}) != row[\'input_sha256\']:\n                raise ValueError(\'Original text hash mismatch\')\n            if any(row[k] != labels[uid][k] for k in (\'student_id\',\'conversation_id\')):\n                raise ValueError(\'CSV/source identity mismatch\')\n            sources[uid] = row\n    if set(sources) != set(labels):\n        raise ValueError(\'CSV ID missing from original source\')\n    # Fixed label-independent policy. Never move IDs or inspect test predictions.\n    seen, excluded, kept = {}, [], {}\n    for split in (\'test\',\'valid\',\'train\'):\n        kept[split] = []\n        for uid in splits[split]:\n            digest = sources[uid][\'input_sha256\']\n            if digest in seen:\n                excluded.append({\'id\':uid,\'split\':split,\'duplicate_of\':seen[digest], \'reason\':\'exact_input_duplicate\'})\n            else:\n                seen[digest] = uid\n                kept[split].append(uid)\n    if any(not kept[split] for split in (\'train\',\'valid\',\'test\')):\n        raise ValueError(\'A split is empty after duplicate removal\')\n    return sources, labels, kept, excluded\n\n\ndef load_cache(config):\n    import torch\n    for name,digest in config[\'inputs\'].items():\n        if sha(Path(config[\'data_dir\'])/name) != digest:\n            raise ValueError(\'Frozen source snapshot changed\')\n    manifest = json.loads((Path(config[\'run_dir\'])/\'token_cache_manifest.json\').read_text(encoding=\'utf-8\'))\n    path = Path(config[\'run_dir\'])/\'tokenized.pt\'\n    if manifest[\'identity\'] != config[\'identity\'] or sha(path) != manifest[\'sha256\']:\n        raise ValueError(\'Token cache identity/content changed\')\n    return torch.load(path,map_location=\'cpu\',weights_only=True)\n', 'mmtrain/evaluate.py': '"""Final held-out evaluation with fixed validation thresholds and per-row predictions."""\n\nimport json\nfrom pathlib import Path\n\nimport numpy as np\n\nfrom .checkpoint import load\nfrom .data import load_cache, sha\nfrom .metrics import evaluate\nfrom .runtime import initialize, predict\n\n\ndef run(config):\n    root = Path(config[\'run_dir\'])\n    committed = load(root/\'latest.pt\',config[\'identity\'])\n    best_name = committed[\'best_file\']\n    if best_name is None:\n        raise ValueError(\'No completed validation checkpoint; finish training first\')\n    del committed\n    best_path, report_path = root/best_name,root/\'test_report.json\'\n    digest = sha(best_path)\n    if report_path.exists():\n        report = json.loads(report_path.read_text(encoding=\'utf-8\'))\n        if report[\'checkpoint_sha256\'] != digest or report[\'identity\'] != config[\'identity\']:\n            raise ValueError(\'Existing test report is for a different checkpoint/config\')\n        print(\'Reusing completed test report\',json.dumps(report,indent=2))\n        return\n    state = load(best_path,config[\'identity\'])\n    cache = load_cache(config)\n    model,_ = initialize(config)\n    model.load_state_dict(state[\'model\'])\n    records, ids = cache[\'records\'],cache[\'splits\'][\'test\']\n    target, probabilities, seconds = predict(model,records,ids,config[\'record_batch\'])\n    report = evaluate(target,probabilities,state[\'thresholds\'])\n    prevalence = np.mean([records[u][\'labels\'] for u in cache[\'splits\'][\'train\']],axis=0)\n    report[\'majority_baseline\'] = evaluate(target,np.tile(prevalence,(len(ids),1)),[.5,.5])\n    report.update(identity=config[\'identity\'],checkpoint_sha256=digest,predict_seconds=seconds,\n                  rows_per_second=len(ids)/max(seconds,1e-9),valid_report=state[\'valid_report\'])\n    temp = root/\'test_predictions.tmp\'\n    with temp.open(\'w\',encoding=\'utf-8\') as stream:\n        for uid,truth,prob in zip(ids,target,probabilities,strict=True):\n            stream.write(json.dumps({\'id\':uid,\'input_sha256\':records[uid][\'input_sha256\'],\n                \'target\':truth.tolist(),\'probabilities\':prob.tolist(),\n                \'prediction\':(prob>=np.asarray(state[\'thresholds\'])).tolist()})+\'\\n\')\n    temp.replace(root/\'test_predictions.jsonl\')\n    temporary = report_path.with_suffix(\'.tmp\')\n    temporary.write_text(json.dumps(report,indent=2),encoding=\'utf-8\')\n    temporary.replace(report_path)\n    print(json.dumps(report,indent=2))\n\n\nif __name__==\'__main__\':\n    import sys\n    run(json.loads(Path(sys.argv[1]).read_text(encoding=\'utf-8\')))\n', 'mmtrain/metrics.py': '"""Binary-class macro-F1 for each task; thresholds selected on validation only."""\n\nimport numpy as np\n\nfrom .data import LABELS\n\n\ndef binary_metrics(target, prediction):\n    target, prediction = np.asarray(target,dtype=bool),np.asarray(prediction,dtype=bool)\n    tp,tn = int(np.sum(target&prediction)),int(np.sum(~target&~prediction))\n    fp,fn = int(np.sum(~target&prediction)),int(np.sum(target&~prediction))\n    def ratio(a,b):\n        return a/b if b else 0.\n    positive_f1, negative_f1 = ratio(2*tp,2*tp+fp+fn),ratio(2*tn,2*tn+fp+fn)\n    return {\'macro_f1\':(positive_f1+negative_f1)/2,\'positive_f1\':positive_f1,\n            \'precision\':ratio(tp,tp+fp),\'recall\':ratio(tp,tp+fn),\'fpr\':fp/(fp+tn) if fp+tn else None,\n            \'tp\':tp,\'tn\':tn,\'fp\':fp,\'fn\':fn,\'positive_support\':tp+fn,\'negative_support\':tn+fp}\n\n\ndef choose_thresholds(target, probabilities):\n    target, probabilities = np.asarray(target),np.asarray(probabilities)\n    result = []\n    for i in range(2):\n        options = [(binary_metrics(target[:,i],probabilities[:,i]>=t)[\'macro_f1\'],-abs(t-.5),-t,t)\n                   for t in np.linspace(.1,.9,17)]\n        result.append(float(max(options)[-1]))\n    return result\n\n\ndef evaluate(target, probabilities, thresholds):\n    target, probabilities = np.asarray(target),np.asarray(probabilities)\n    if target.shape != probabilities.shape or target.ndim!=2 or target.shape[1]!=2 or len(target)==0:\n        raise ValueError(\'Expected nonempty two-label arrays\')\n    prediction = probabilities>=np.asarray(thresholds)\n    labels = {key:binary_metrics(target[:,i],prediction[:,i]) for i,key in enumerate(LABELS)}\n    return {\'labels\':labels,\'mean_label_macro_f1\':float(np.mean([v[\'macro_f1\'] for v in labels.values()])),\n            \'exact_match\':float(np.mean(np.all(target.astype(bool)==prediction,axis=1))),\n            \'thresholds\':list(thresholds),\'rows\':len(target),\n            \'interpretation\':\'Agreement with automatically verified Qwen labels, not human-gold accuracy.\'}\n', 'mmtrain/model.py': '"""Full-token window encoder plus current-conditioned context and student pooling."""\n\nimport torch\nfrom torch import nn\nfrom torch.utils.checkpoint import checkpoint\n\n\ndef pad_windows(chunks, pad_id, device):\n    length = max(len(c[\'input_ids\']) for c in chunks)\n    result = {}\n    for key in (\'input_ids\',\'student_mask\',\'current_mask\',\'owner_mask\'):\n        fill = pad_id if key==\'input_ids\' else 0\n        result[key] = torch.tensor([c[key]+[fill]*(length-len(c[key])) for c in chunks],device=device)\n    result[\'attention_mask\'] = torch.tensor([[1]*len(c[\'input_ids\'])+[0]*(length-len(c[\'input_ids\']))\n                                            for c in chunks],device=device)\n    return result\n\n\nclass DialogueClassifier(nn.Module):\n    def __init__(self, encoder, hidden, pad_id, window_batch=2, dropout=.1):\n        super().__init__()\n        self.encoder, self.pad_id, self.window_batch = encoder, pad_id, window_batch\n        self.position = nn.Linear(3,hidden)\n        self.context_key = nn.Linear(hidden,hidden)\n        self.student_key = nn.Linear(hidden,hidden)\n        self.query_key = nn.Linear(hidden,hidden)\n        self.context_score = nn.Linear(hidden,1)\n        self.student_score = nn.Linear(hidden,1)\n        self.heads = nn.ModuleList([nn.Sequential(nn.LayerNorm(hidden*4),nn.Dropout(dropout),\n                                    nn.Linear(hidden*4,hidden),nn.GELU(),nn.Linear(hidden,1)) for _ in range(2)])\n\n    def encode_pool(self, ids, attention, student, current, owner):\n        hidden = self.encoder(input_ids=ids,attention_mask=attention).last_hidden_state.float()\n        def pool(mask):\n            mask = mask.unsqueeze(-1).to(hidden.dtype)\n            return (hidden*mask).sum(1)/mask.sum(1).clamp(min=1)\n        return torch.stack([pool(owner),pool(student),pool(current)],dim=1)\n\n    def encode_chunks(self, chunks):\n        device = next(self.parameters()).device\n        features = []\n        for start in range(0,len(chunks),self.window_batch):\n            batch = pad_windows(chunks[start:start+self.window_batch],self.pad_id,device)\n            args = [batch[k] for k in (\'input_ids\',\'attention_mask\',\'student_mask\',\'current_mask\',\'owner_mask\')]\n            if self.training and torch.is_grad_enabled():\n                features.append(checkpoint(self.encode_pool,*args,use_reentrant=False,preserve_rng_state=True))\n            else:\n                features.append(self.encode_pool(*args))\n        return torch.cat(features).float()\n\n    def classify(self, chunks, features, return_attention=False):\n        device = features.device\n        current_counts = torch.tensor([sum(c[\'current_mask\']) for c in chunks],device=device,dtype=torch.float32)\n        student_counts = torch.tensor([sum(c[\'student_mask\']) for c in chunks],device=device,dtype=torch.float32)\n        owner_counts = torch.tensor([sum(c[\'owner_mask\']) for c in chunks],device=device,dtype=torch.float32)\n        query = (features[:,2]*current_counts[:,None]).sum(0)/current_counts.sum().clamp(min=1)\n        positions = torch.stack([torch.arange(len(chunks),device=device)/max(len(chunks)-1,1),\n                                current_counts/owner_counts.clamp(min=1),\n                                student_counts/owner_counts.clamp(min=1)],dim=1)\n        position = self.position(positions)\n        def attend(values, key, score, available):\n            scores = score(torch.tanh(key(values)+self.query_key(query)[None]+position)).squeeze(-1)\n            weights = torch.softmax(scores.masked_fill(~available,-1e9),dim=0)\n            weights = weights*available\n            weights = weights/weights.sum().clamp(min=1e-12)\n            return (weights[:,None]*values).sum(0),weights\n        context, ca = attend(features[:,0],self.context_key,self.context_score,owner_counts>0)\n        student, sa = attend(features[:,1],self.student_key,self.student_score,student_counts>0)\n        logits = []\n        for head, pooled in zip(self.heads,(context,student),strict=True):\n            combined = torch.cat([query,pooled,query*pooled,torch.abs(query-pooled)])\n            logits.append(head(combined).squeeze(-1))\n        logits = torch.stack(logits)\n        return (logits,{\'context\':ca,\'student\':sa}) if return_attention else logits\n\n    def forward(self, chunks, return_attention=False):\n        return self.classify(chunks,self.encode_chunks(chunks),return_attention)\n\n    def forward_many(self, bags):\n        flat = [chunk for bag in bags for chunk in bag]\n        features = self.encode_chunks(flat)\n        outputs, start = [], 0\n        for bag in bags:\n            outputs.append(self.classify(bag,features[start:start+len(bag)]))\n            start += len(bag)\n        return torch.stack(outputs)\n\n\ndef build_model(config):\n    from transformers import AutoConfig, AutoModel\n    hf = AutoConfig.from_pretrained(config[\'model_id\'],revision=config[\'revision\'])\n    if hf.max_position_embeddings < config[\'chunk_length\']:\n        raise ValueError(\'Chunk exceeds pretrained context length\')\n    hf.reference_compile = False\n    encoder = AutoModel.from_pretrained(config[\'model_id\'],revision=config[\'revision\'],config=hf,\n                                        attn_implementation=\'sdpa\')\n    encoder.float()\n    return DialogueClassifier(encoder,hf.hidden_size,config[\'pad_id\'],config[\'window_batch\'])\n', 'mmtrain/prepare.py': '"""Lossless token cache plus coverage, length, duplicate and class reports."""\n\nimport json\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\n\nfrom .chunks import make_chunks\nfrom .data import LABELS, load_cache, load_sources, sha\n\n\ndef prepare(config):\n    from transformers import AutoTokenizer\n    root = Path(config[\'run_dir\'])\n    if (root/\'token_cache_manifest.json\').exists():\n        cache = load_cache(config)\n        print(\'Reusing token cache:\',{k:len(v) for k,v in cache[\'splits\'].items()},flush=True)\n        return\n    for name,digest in config[\'inputs\'].items():\n        if sha(Path(config[\'data_dir\'])/name) != digest:\n            raise ValueError(\'Input changed after configuration was frozen\')\n    sources, labels, splits, excluded = load_sources(config[\'data_dir\'])\n    tokenizer = AutoTokenizer.from_pretrained(config[\'model_id\'],revision=config[\'revision\'])\n    records, split_report = {}, {}\n    for split, ids in splits.items():\n        lengths, windows = [], []\n        for i,uid in enumerate(ids):\n            chunks, stats = make_chunks(sources[uid],tokenizer,config[\'chunk_length\'],config[\'overlap\'])\n            row = {k:sources[uid][k] for k in (\'dataset\',\'source_id\',\'student_id\',\'conversation_id\',\'input_sha256\')}\n            row.update(chunks=chunks,labels=[float(labels[uid][k]==\'true\') for k in LABELS],stats=stats)\n            records[uid] = row\n            lengths.append(stats[\'single_input_tokens\'])\n            windows.append(stats[\'chunks\'])\n            if (i+1)%250==0:\n                print(split,\'tokenized\',i+1,\'/\',len(ids),flush=True)\n        split_report[split] = {\'rows\':len(ids),\'tokens_percentiles\':dict(zip((\'p50\',\'p90\',\'p95\',\'p99\',\'max\'),\n                                  np.percentile(lengths,[50,90,95,99,100]).astype(int).tolist(),strict=True)),\n                              \'over_8192\':sum(n>8192 for n in lengths),\'max_chunks\':max(windows),\n                              \'total_chunks\':sum(windows),\n                              \'positives\':{key:int(sum(records[uid][\'labels\'][j] for uid in ids))\n                                           for j,key in enumerate(LABELS)}}\n    cache = {\'records\':records,\'splits\':splits,\'excluded\':excluded,\'identity\':config[\'identity\']}\n    temp = root/\'tokenized.tmp\'\n    torch.save(cache,temp)\n    temp.replace(root/\'tokenized.pt\')\n    manifest = {\'identity\':config[\'identity\'],\'sha256\':sha(root/\'tokenized.pt\'),\'splits\':split_report,\n                \'excluded_exact_duplicates\':excluded,\'original_csv_rows\':len(sources),\n                \'kept_rows\':len(records),\'all_tokens_processed\':True,\'max_chunks_cap\':None}\n    (root/\'token_cache_manifest.json\').write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding=\'utf-8\')\n    print(json.dumps({k:v for k,v in manifest.items() if k!=\'excluded_exact_duplicates\'},ensure_ascii=False,indent=2))\n\n\nif __name__ == \'__main__\':\n    import sys\n    prepare(json.loads(Path(sys.argv[1]).read_text(encoding=\'utf-8\')))\n', 'mmtrain/runtime.py': '"""GPU initialization, equal-record loss and prediction independent of test selection."""\n\nimport contextlib\nimport time\n\nimport numpy as np\nimport torch\nfrom torch import nn\n\nfrom .model import build_model\n\n\ndef initialize(config):\n    if not torch.cuda.is_available() or not torch.cuda.is_bf16_supported():\n        raise RuntimeError(\'A100/H100 BF16 GPU required\')\n    torch.manual_seed(config[\'seed\'])\n    torch.cuda.manual_seed_all(config[\'seed\'])\n    print(\'GPU:\',torch.cuda.get_device_name(0),\'window batch:\',config[\'window_batch\'],\n          \'record batch:\',config[\'record_batch\'],flush=True)\n    model = build_model(config).cuda()\n    encoder_ids = {id(p) for p in model.encoder.parameters()}\n    optimizer = torch.optim.AdamW([\n        {\'params\':[p for p in model.parameters() if id(p) in encoder_ids], \'lr\':config[\'encoder_lr\']},\n        {\'params\':[p for p in model.parameters() if id(p) not in encoder_ids], \'lr\':config[\'head_lr\']}],\n        weight_decay=.01)\n    return model,optimizer\n\n\ndef amp(model):\n    return torch.autocast(\'cuda\',dtype=torch.bfloat16) if next(model.parameters()).is_cuda else contextlib.nullcontext()\n\n\ndef loss_group(model, records, ids, record_batch, pos_weight=None):\n    """One equally weighted loss per conversation, regardless of number of windows."""\n    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)\n    device = next(model.parameters()).device\n    total = 0.\n    for start in range(0,len(ids),record_batch):\n        current = ids[start:start+record_batch]\n        target = torch.tensor([records[uid][\'labels\'] for uid in current],device=device)\n        with amp(model):\n            logits = model.forward_many([records[uid][\'chunks\'] for uid in current])\n            loss = criterion(logits,target)*(len(current)/len(ids))\n        if not torch.isfinite(loss):\n            raise ValueError(\'Nonfinite training loss\')\n        loss.backward()\n        total += float(loss.detach())\n    return total\n\n\ndef predict(model, records, ids, record_batch):\n    model.eval()\n    probabilities, target = [], []\n    if next(model.parameters()).is_cuda:\n        torch.cuda.synchronize()\n    started = time.monotonic()\n    with torch.inference_mode():\n        for start in range(0,len(ids),record_batch):\n            current = ids[start:start+record_batch]\n            with amp(model):\n                logits = model.forward_many([records[uid][\'chunks\'] for uid in current])\n            probabilities.extend(torch.sigmoid(logits.float()).cpu().tolist())\n            target.extend(records[uid][\'labels\'] for uid in current)\n    if next(model.parameters()).is_cuda:\n        torch.cuda.synchronize()\n    return np.asarray(target),np.asarray(probabilities),time.monotonic()-started\n', 'mmtrain/train.py': '"""Full encoder fine-tuning, validation-only selection, exact optimizer-boundary resume."""\n\nimport argparse\nimport json\nimport math\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\n\nfrom .checkpoint import load, restore_rng, rng_state, save\nfrom .data import load_cache\nfrom .metrics import choose_thresholds, evaluate\nfrom .runtime import initialize, loss_group, predict\n\n\ndef epoch_ids(ids, records, epoch, seed):\n    generator = torch.Generator().manual_seed(seed+epoch)\n    ordered = sorted(ids,key=lambda uid:records[uid][\'stats\'][\'chunks\'])\n    buckets = [ordered[i:i+64] for i in range(0,len(ordered),64)]\n    result = []\n    for index in torch.randperm(len(buckets),generator=generator).tolist():\n        bucket = buckets[index]\n        result.extend(bucket[j] for j in torch.randperm(len(bucket),generator=generator).tolist())\n    return result\n\n\ndef train(config, smoke=False):\n    cache = load_cache(config)\n    records, splits = cache[\'records\'],cache[\'splits\']\n    root = Path(config[\'run_dir\'])\n    model, optimizer = initialize(config)\n    train_ids = splits[\'train\']\n    prevalence = np.mean([records[uid][\'labels\'] for uid in train_ids],axis=0)\n    weights = np.clip((1-prevalence)/np.maximum(prevalence,1/len(train_ids)),.5,5)\n    pos_weight = torch.tensor(weights,device=\'cuda\',dtype=torch.float32) if config[\'class_weights\'] else None\n    if smoke:\n        chosen = sorted(train_ids,key=lambda uid:records[uid][\'stats\'][\'chunks\'])\n        chosen = list(dict.fromkeys(chosen[:2]+chosen[-1:]))\n        started = time.monotonic()\n        model.train()\n        optimizer.zero_grad(set_to_none=True)\n        loss = loss_group(model,records,chosen,config[\'record_batch\'],pos_weight)\n        if not any(p.grad is not None and torch.count_nonzero(p.grad).item() for p in model.encoder.parameters()):\n            raise ValueError(\'Encoder gradients missing\')\n        torch.nn.utils.clip_grad_norm_(model.parameters(),1.)\n        optimizer.step()\n        torch.cuda.synchronize()\n        report = {\'loss\':loss,\'sample_rows\':len(chosen),\n                  \'max_sample_chunks\':max(records[u][\'stats\'][\'chunks\'] for u in chosen),\n                  \'seconds\':time.monotonic()-started,\'peak_vram_gib\':torch.cuda.max_memory_allocated()/2**30,\n                  \'note\':\'Smoke includes the longest train conversation; weights are NOT used for full training.\'}\n        (root/\'smoke_report.json\').write_text(json.dumps(report,indent=2),encoding=\'utf-8\')\n        print(json.dumps(report,indent=2))\n        return\n    steps_per_epoch = math.ceil(len(train_ids)/config[\'effective_batch\'])\n    total = steps_per_epoch*config[\'epochs\']\n    warmup = max(1,int(total*.1))\n    def multiplier(step):\n        return step/warmup if step<warmup else .5*(1+math.cos(math.pi*(step-warmup)/max(1,total-warmup)))\n    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer,multiplier)\n    epoch,offset,step,best,bad_epochs,best_file = 0,0,0,-1.,0,None\n    latest = root/\'latest.pt\'\n    if latest.exists():\n        state = load(latest,config[\'identity\'])\n        model.load_state_dict(state[\'model\'])\n        optimizer.load_state_dict(state[\'optimizer\'])\n        scheduler.load_state_dict(state[\'scheduler\'])\n        epoch,offset,step,best,bad_epochs = [state[k] for k in (\'epoch\',\'offset\',\'step\',\'best\',\'bad_epochs\')]\n        best_file = state[\'best_file\']\n        restore_rng(state[\'rng\'])\n        del state\n        print(\'Resuming optimizer step\',step,\'epoch\',epoch,\'next row offset\',offset,flush=True)\n    def checkpoint(next_epoch, next_offset):\n        save(latest,{\'identity\':config[\'identity\'],\'model\':model.state_dict(),\'optimizer\':optimizer.state_dict(),\n                     \'scheduler\':scheduler.state_dict(),\'rng\':rng_state(),\'epoch\':next_epoch,\'offset\':next_offset,\n                     \'step\':step,\'best\':best,\'bad_epochs\':bad_epochs,\'best_file\':best_file})\n        print(\'Drive checkpoint committed:\',step,flush=True)\n    if bad_epochs >= config[\'patience\']:\n        print(\'Early stopping already satisfied; using best.pt\',flush=True)\n        return\n    while epoch<config[\'epochs\']:\n        ordered = epoch_ids(train_ids,records,epoch,config[\'seed\'])\n        model.train()\n        started = time.monotonic()\n        for start in range(offset,len(ordered),config[\'effective_batch\']):\n            group = ordered[start:start+config[\'effective_batch\']]\n            optimizer.zero_grad(set_to_none=True)\n            loss = loss_group(model,records,group,config[\'record_batch\'],pos_weight)\n            norm = torch.nn.utils.clip_grad_norm_(model.parameters(),1.)\n            if not torch.isfinite(norm):\n                raise ValueError(\'Nonfinite gradient norm\')\n            optimizer.step()\n            scheduler.step()\n            step += 1\n            if step%10==0:\n                print(\'epoch\',epoch+1,\'step\',step,\'rows\',start+len(group),\'/\',len(ordered),\n                      \'loss\',round(loss,4),\'elapsed sec\',round(time.monotonic()-started,1),flush=True)\n            if step%config[\'save_steps\']==0 and start+len(group)<len(ordered):\n                checkpoint(epoch,start+len(group))\n        target, probabilities, seconds = predict(model,records,splits[\'valid\'],config[\'record_batch\'])\n        thresholds = choose_thresholds(target,probabilities)\n        report = evaluate(target,probabilities,thresholds)\n        report.update(epoch=epoch+1,optimizer_step=step,predict_seconds=seconds,\n                      peak_vram_gib=torch.cuda.max_memory_allocated()/2**30)\n        score = report[\'mean_label_macro_f1\']\n        if score>best:\n            best,bad_epochs = score,0\n            best_file = f\'best_step_{step}.pt\'\n            save(root/best_file,{\'identity\':config[\'identity\'],\'model\':model.state_dict(),\n                                 \'thresholds\':thresholds,\'valid_report\':report})\n            (root/\'best_valid_report.json\').write_text(json.dumps(report,indent=2),encoding=\'utf-8\')\n        else:\n            bad_epochs += 1\n        with (root/\'training_log.jsonl\').open(\'a\',encoding=\'utf-8\') as stream:\n            stream.write(json.dumps(report)+\'\\n\')\n        print(json.dumps(report,indent=2),flush=True)\n        epoch,offset = epoch+1,0\n        checkpoint(epoch,0)\n        if bad_epochs>=config[\'patience\']:\n            print(\'Early stopping on VALID only\',flush=True)\n            break\n    print(\'Training complete; committed best model and thresholds:\',root/best_file,flush=True)\n\n\nif __name__==\'__main__\':\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'config\')\n    parser.add_argument(\'--smoke\',action=\'store_true\')\n    args = parser.parse_args()\n    train(json.loads(Path(args.config).read_text(encoding=\'utf-8\')),args.smoke)\n'}
for relative,text in SOURCES.items():
    path = CODE/relative
    path.parent.mkdir(parents=True,exist_ok=True)
    path.write_text(text,encoding='utf-8')
if str(CODE) not in sys.path:
    sys.path.insert(0,str(CODE))
for name in list(sys.modules):
    if name=='mmtrain' or name.startswith('mmtrain.'):
        del sys.modules[name]
print('계층형 mmBERT 코드 준비 완료')


## 4. 모델·길이·학습 설정 고정

In [ ]:
import importlib.metadata
import os
import subprocess
import sys
from huggingface_hub import HfApi
from transformers import AutoTokenizer
from mmtrain.data import sha, stable

SEED = 42
MODEL_ID = 'jhu-clsp/mmBERT-base'
CHUNK_LENGTH = 8192       # 전체 대화 길이 제한이 아닌, 한 구간의 최대 길이.
OVERLAP = 256
WINDOW_BATCH = 2          # 인코더에 동시에 올릴 구간 수. A100/H100에서 smoke 후 4/8 검토.
RECORD_BATCH = 2          # 한 번에 처리할 학생 발화 샘플 수. 각 샘플의 이전 대화도 전부 유지.
EFFECTIVE_BATCH = 16      # optimizer 갱신 한 번당 샘플 수. 긴 샘플도 동일한 가중치.
EPOCHS = 3
ENCODER_LR = 2e-5
HEAD_LR = 1e-4
SAVE_STEPS = 100          # optimizer 포함 수 GB. Drive latest를 원자적으로 교체.
CLASS_WEIGHTS = True      # train에서만 계산하고 0.5~5로 제한.
if not 16 <= CHUNK_LENGTH <= 8192 or not 0 <= OVERLAP < CHUNK_LENGTH-8:
    raise ValueError('구간 길이·overlap 설정을 확인하세요.')
if min(WINDOW_BATCH,RECORD_BATCH,EFFECTIVE_BATCH,EPOCHS,SAVE_STEPS) < 1:
    raise ValueError('배치·epoch·저장 주기는 양수여야 합니다.')

pin_path = RUN_DIR/'model_pin.json'
if pin_path.exists():
    pin = json.loads(pin_path.read_text(encoding='utf-8'))
    if pin['model_id'] != MODEL_ID:
        raise ValueError('모델 변경 시 RUN_NAME도 바꾸세요.')
else:
    pin = {'model_id':MODEL_ID,'revision':HfApi().model_info(MODEL_ID).sha}
    pin_path.write_text(json.dumps(pin,indent=2),encoding='utf-8')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID,revision=pin['revision'])
if tokenizer.pad_token_id is None:
    raise ValueError('Padding token이 필요합니다.')
config = {'run_dir':str(RUN_DIR),'data_dir':str(DATA_DIR),'model_id':MODEL_ID,'revision':pin['revision'],
          'seed':SEED,'pad_id':tokenizer.pad_token_id,'chunk_length':CHUNK_LENGTH,'overlap':OVERLAP,
          'window_batch':WINDOW_BATCH,'record_batch':RECORD_BATCH,'effective_batch':EFFECTIVE_BATCH,
          'epochs':EPOCHS,'encoder_lr':ENCODER_LR,'head_lr':HEAD_LR,'save_steps':SAVE_STEPS,
          'class_weights':CLASS_WEIGHTS,'patience':2,
          'inputs':{p.name:sha(p) for p in sorted(DATA_DIR.iterdir()) if p.is_file()},
          'versions':VERSIONS,'implementation':{str(p.relative_to(CODE)):sha(p) for p in sorted(CODE.rglob('*.py'))}}
SCHEDULE_KEYS = ('window_batch','record_batch','save_steps')
config['identity'] = stable({k:v for k,v in config.items() if k not in SCHEDULE_KEYS})
CONFIG_PATH = RUN_DIR/'config.json'
if CONFIG_PATH.exists():
    previous = json.loads(CONFIG_PATH.read_text(encoding='utf-8'))
    if previous['identity'] != config['identity']:
        raise ValueError('실험 설정이 다릅니다. RUN_NAME을 새로 지정하세요.')
    if previous != config:
        with (RUN_DIR/'schedule_changes.jsonl').open('a',encoding='utf-8') as stream:
            stream.write(json.dumps({k:config[k] for k in SCHEDULE_KEYS})+'\n')
CONFIG_PATH.write_text(json.dumps(config,indent=2),encoding='utf-8')
del tokenizer

def job(module,*args):
    environment = dict(os.environ,PYTHONPATH=str(CODE),PYTHONUNBUFFERED='1',TOKENIZERS_PARALLELISM='true')
    process = subprocess.Popen([sys.executable,'-m',module,str(CONFIG_PATH),*args],cwd=CODE,env=environment,
                               stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
    try:
        for line in process.stdout:
            print(line,end='',flush=True)
        if process.wait()!=0:
            raise RuntimeError('처리 실패. 위 오류를 확인하세요. 기존 Drive 파일은 지우지 마세요.')
    except BaseException:
        process.terminate()
        try:
            process.wait(timeout=20)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
        raise

print('mmBERT full fine-tuning입니다. LoRA가 아닙니다. 토큰 캐시 후 smoke를 실행하세요.')


## 5. 원문 연결·전체 토큰화·길이/중복/분포 검사
토큰화는 한 번만 수행하고 Drive에 저장합니다. 누락 없이 전체 구간을 처리합니다.

In [ ]:
job('mmtrain.prepare')


## 6. GPU smoke — 짧은 대화와 가장 긴 train 대화
인코더 gradient와 메모리를 확인합니다. 이 가중치는 전체 학습에 사용하지 않습니다.
OOM이면 4번의 WINDOW_BATCH=1, RECORD_BATCH=1로 낮추고 4번부터 재실행하세요. 토큰 캐시는 재사용됩니다.

In [ ]:
job('mmtrain.train','--smoke')


## 7. 전체 학습·validation 선택·중단 후 재개
같은 셀을 다시 실행하면 마지막 optimizer 체크포인트에서 재개합니다.

In [ ]:
RUN_TRAIN = False
if RUN_TRAIN:
    job('mmtrain.train')
else:
    print('smoke 결과 확인 후 RUN_TRAIN=True로 변경하세요.')


## 8. 마지막 test 평가 — 설정·best 고정 후 한 번만
이 결과로 학습률·길이·모델을 다시 선택하지 마세요. 재실행 시 완료된 동일 보고서를 재사용합니다.

In [ ]:
RUN_TEST = False
if RUN_TEST:
    job('mmtrain.evaluate')
else:
    print('전체 학습 완료 후 RUN_TEST=True로 변경하세요.')


## Drive 결과

ThinkGuard_NLP/studychat_mmbert_hierarchical_v1/
- token_cache_manifest.json: 길이·8192 초과·구간 수·중복 제외·양성 분포. 원문을 버린 건수와 구분하세요.
- tokenized.pt: 원문에서 만든 모든 구간과 라벨. 모델 입력에는 라벨이 포함되지 않습니다.
- smoke_report.json: longest train 처리 여부, 메모리, 샘플 처리 시간. 전체 시간 예측값은 아닙니다.
- latest.pt: 학습 재개용 model/optimizer/scheduler/RNG/위치. 직접 삭제하지 마세요.
- best_step_*.pt: validation으로 선택된 모델 가중치·threshold. latest가 가리키는 committed best 사용.
- best_valid_report.json / training_log.jsonl: validation 선택 기록.
- test_report.json / test_predictions.jsonl: 라벨별 F1·오탐률·정확 일치·속도·train 다수 클래스 기준선.
- config.json / model_pin.json / environment_versions.json: 재현 설정과 모델 commit SHA.

런타임이 사라지면 같은 노트북 1~5번을 복구한 뒤 7번을 실행하세요. 토큰 캐시와 optimizer를 재사용합니다.
학습 완료 후 마지막 평가만 필요하면 1~4번 복구 후 8번만 실행해도 됩니다.
checkpoint 내부에는 Python pickle이 포함되므로 이 노트북이 생성한 파일만 사용하세요.
